# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIR⁲ ([FAIR2](https://sen.science/doi/10.71728/senscience.y7m0-f273)) dataset using the `mlcroissant` library. We demonstrate how to interact with `mlcroissant`-powered data packages by referencing all entities (record sets, fields, columns, etc.) via their `@id` values, as recommended for Croissant schema datasets.

### Dataset Source
The dataset source is provided by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant --quiet

## 1. Data Loading

We use `mlcroissant` to download and parse the FAIR⁲ Croissant package and access all dataset metadata and record sets. This step only loads the schema description and record set definitions, not the entire raw data yet.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
print("--- Dataset Metadata ---")
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview

Below, we list all available `recordSet` entries, their `@id`, and fields with their `@id` as defined in the loaded Croissant schema. This helps to discover what data tables ('record sets') are available, along with the proper IDs to use in downstream `mlcroissant` calls.

The actual record set IDs and field/column IDs can be inspected from the schema.

> **Note:** If you are unsure of the specific record sets available (e.g., if `dataset.metadata.recordSet` is empty), you can access them from `dataset.record_sets()`.

In [ ]:
# List available record sets with their fields/columns using @id

def list_record_sets(ds):
    print("--- Record Sets in Dataset (with @id and field/column @id) ---")
    for record_set in ds.record_sets():
        print(f"RecordSet @id: {record_set['@id']}")
        # Explore fields
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field['@id']}")
                else:
                    print(f"    - {field}")
        # Explore columns (if any columns are defined directly on the record set)
        columns = record_set.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("  Columns:")
            for col in columns:
                if isinstance(col, dict):
                    print(f"    - {col['@id']}")
                else:
                    print(f"    - {col}")
        print("")

list_record_sets(dataset)

## 3. Data Extraction

We extract all data from each record set, referencing all record sets by their `@id`. For each record set, the data is loaded into a pandas DataFrame for easy analysis.

> ⚠️ **Make sure to use only the `@id` string of the record set and fields for all subsequent operations.**

Edit the `record_sets_to_load` list below to specify which record sets to extract, using the `@id` values discovered in the previous section.

In [ ]:
# List all record set @ids (from the previous cell)
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print("Loading the following record sets:")
for rid in record_set_ids:
    print(f"  - {rid}")

# Extract records from each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"  Warning: No records found for {record_set_id}")

# Preview the first loaded DataFrame (if any)
if dataframes:
    first_id = next(iter(dataframes))
    print(f"\nColumns for first record set '{first_id}':\n{dataframes[first_id].columns.tolist()}")
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded. Please check dataset availability.")

## 4. Exploratory Data Analysis (EDA)

This section demonstrates basic filtering and transformation on numeric fields, referencing all columns/fields by `@id`. We filter on a chosen numeric field, normalize it, and show how to group by another field — all using proper Croissant entity IDs.

> You may need to update the numeric or group field variables below to match your actual data. Use the column `@id` names printed in the previous section.

In [ ]:
import numpy as np

# Select the record set to analyze (update this if you want a specific one)
if not dataframes:
    print("No data available for EDA.")
else:
    record_set_id = next(iter(dataframes))  # Pick the first loaded record set for demonstration
    df = dataframes[record_set_id]

    print(f"Running EDA on record set: {record_set_id}\n")
    print("Sample data:")
    display(df.head())

    # Pick numeric field by its column @id (update if needed)
    potential_numeric = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            potential_numeric.append(col)
    print(f"Numeric columns (by @id): {potential_numeric}")
    
    if not potential_numeric:
        print("No numeric columns for demo.")
    else:
        numeric_field_id = potential_numeric[0]  # Take the first
        print(f"Using numeric field: {numeric_field_id}")

        # Simple threshold for demo (could be refined)
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"\nFiltered rows where '{numeric_field_id}' > mean:")
        display(filtered_df[[numeric_field_id]].head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a reasonable grouping field (non-numeric @id column)
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == "O"]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization

We will plot a histogram of the selected numeric field and, if a grouping field is available, a boxplot by group. All axis/data references use schema `@id` column names.

In [ ]:
import matplotlib.pyplot as plt

if not dataframes:
    print("No data to visualize.")
else:
    df = dataframes[record_set_id]

    if potential_numeric:
        numeric_field = numeric_field_id
        plt.figure(figsize=(8, 4))
        df[numeric_field].dropna().hist(bins=30, alpha=0.7)
        plt.title(f"Histogram of '{numeric_field}'")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()
        
        # Boxplot if we have a group
        if group_candidates:
            group_field = group_candidates[0]
            plt.figure(figsize=(10, 5))
            df.boxplot(column=numeric_field, by=group_field, rot=45)
            plt.title(f"Boxplot of '{numeric_field}' by '{group_field}'")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    else:
        print("No numeric column to plot.")

## 6. Conclusion

- Using the Croissant schema, all data exploration and analysis referenced entities by their `@id`, ensuring reproducibility and schema alignment.
- You can further analyze the dataframes, filter, or visualize any additional record sets as needed for your use case.
- For complete reproducibility, match any application-specific fields using their `@id` directly from the Croissant schema or by inspecting the dataframes as above.

### Key Takeaway
The `mlcroissant` library enables portable and schema-aware data exploration on FAIR datasets, keeping all references explicit at the metadata level for robust, documented workflows.